In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

N = 2000
timestamps = pd.date_range(start="2025-01-01", periods=N, freq="h")

rows = []

for t in timestamps:

    hour = t.hour

    # normal voltage & current
    voltage = np.random.uniform(210, 240)
    current = np.random.uniform(0.5, 5)

    # normal consumption
    consumption = voltage * current / 1000

    # -----------------------------
    # Inject anomaly (5%)
    # -----------------------------
    if np.random.rand() < 0.05:
        current *= np.random.uniform(2, 4)   # abnormal spike
        consumption = voltage * current / 1000
        anomaly = 1
    else:
        anomaly = 0

    rows.append([
        t,
        hour,
        voltage,
        current,
        round(consumption, 2),
        anomaly
    ])

df = pd.DataFrame(rows, columns=[
    "timestamp",
    "hour",
    "voltage",
    "current",
    "consumption",
    "anomaly"   # for testing only
])

df.to_csv("data/anomaly.csv", index=False)

print("🚨 Anomaly dataset created!")

🚨 Anomaly dataset created!


In [3]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import joblib

df = pd.read_csv("data/anomaly.csv")

features = [
    "hour",
    "voltage",
    "current",
    "consumption"
]

X = df[features]

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(X)

joblib.dump(model, "models/anomaly.pkl")

print("🚨 Anomaly model trained!")

🚨 Anomaly model trained!


In [4]:
from sklearn.metrics import accuracy_score

y_true = df["anomaly"]
y_pred = model.predict(X)
y_pred_binary = [1 if p == -1 else 0 for p in y_pred]
acc = accuracy_score(y_true, y_pred_binary)
print("Accuracy:", round(acc*100, 2), "%")

Accuracy: 96.55 %


In [8]:
import joblib
import pandas as pd

model = joblib.load("models/anomaly.pkl")

# -----------------------------
# INPUT (simulate real case)
# -----------------------------
input_data = {
    "hour": 19,
    "voltage": 230,
    "current": .1 ,   # 🔥 suspicious high
    "consumption": (230 * .1) / 1000
}

print("input_data:", input_data)

df = pd.DataFrame([input_data])

prediction = model.predict(df)[0]

# -----------------------------
# OUTPUT
# -----------------------------
if prediction == -1:
    print("\n🚨 ANOMALY DETECTED (Possible Theft!)")
else:
    print("\n✅ Normal Usage")

input_data: {'hour': 19, 'voltage': 230, 'current': 0.1, 'consumption': 0.023}

🚨 ANOMALY DETECTED (Possible Theft!)


In [8]:
import joblib
import pandas as pd

model = joblib.load("models/anomaly.pkl")

tests = [
    ("Normal Usage", 2),
    ("Slight Increase", 4),
    ("High Usage", 8),
    ("Extreme Spike", 12)
]

for name, current in tests:

    voltage = 230
    consumption = (voltage * current) / 1000

    df = pd.DataFrame([{
        "hour": 20,
        "voltage": voltage,
        "current": current,
        "consumption": consumption
    }])

    pred = model.predict(df)[0]

    print(f"\n{name}")
    print("Result:", "🚨 Anomaly" if pred == -1 else "✅ Normal")


Normal Usage
Result: ✅ Normal

Slight Increase
Result: ✅ Normal

High Usage
Result: 🚨 Anomaly

Extreme Spike
Result: 🚨 Anomaly


In [5]:
import requests

url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "lat": 11.92237,
    "lon": 79.60673,
    "appid": "c8c56b9a7de5112b69a94159a138bec2",
    "units": "metric"
}

response = requests.get(url, params=params)
data = response.json()
print(data)

{'coord': {'lon': 79.6067, 'lat': 11.9224}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02n'}], 'base': 'stations', 'main': {'temp': 27.83, 'feels_like': 31.97, 'temp_min': 27.83, 'temp_max': 27.83, 'pressure': 1010, 'humidity': 82, 'sea_level': 1010, 'grnd_level': 1006}, 'visibility': 10000, 'wind': {'speed': 3.53, 'deg': 150, 'gust': 7.41}, 'clouds': {'all': 11}, 'dt': 1776186836, 'sys': {'type': 2, 'id': 2105418, 'country': 'IN', 'sunrise': 1776126621, 'sunset': 1776171186}, 'timezone': 19800, 'id': 1253490, 'name': 'Valavanur', 'cod': 200}


In [2]:
print(1)

1
